In [1]:

from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais
from pipelines.readers.cvm_balanco_patrimonial import ReaderCSVCVMBalancoPatrimonial

from yfinance import download
from numpy import nan, arange, polyfit, polyval
from pandas import DataFrame, set_option, concat, DateOffset
from datetime import date
from dateutil.relativedelta import relativedelta

In [2]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-13,VIVT3,TELEF BRASIL,ON,707125712,1.181
69,IBEP,2026-07-13,WEGE3,WEG,ON NM,1485954732,3.228
70,IBEP,2026-07-13,YDUQ3,YDUQS PART,ON NM,261365845,0.110


In [3]:
tickers = list(df_b3_segmentos_setoriais["Código"]) + ["^BVSP"]

In [ ]:

reader_csv_balanco_patrimonial = ReaderCSVCVMBalancoPatrimonial()

data = {}

benchmark = download("^BVSP", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)[["Adj Close"]].pct_change()

for codigo in tickers:
    
    # Balanco Patrimonial
    try:
        ativo_total = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="BPA_ind", cd_conta="1").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        ativo_total = nan
    try:
        resultado_bruto = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="DRE_ind", cd_conta="3.03").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        resultado_bruto = nan    
    try:
        receita_de_vendas = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="DRE_ind", cd_conta="3.01").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        receita_de_vendas = nan

    try:
        serie_precos = download(codigo + ".SA" if codigo != "^BVSP" else codigo, period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)
    except Exception:
        
        drawdown_atual = nan
        residuo_reg_lin = nan

        continue
    
    # Drawdown
    preco_maximo = serie_precos['Adj Close'].max()
    preco_atual = serie_precos['Adj Close'].iloc[-1]
    drawdown_atual = (preco_atual - preco_maximo) / preco_maximo * 100
    
    # Regressão linear
    y = serie_precos["Adj Close"].to_numpy()
    x = arange(len(y))

    coef = polyfit(x, y, 1)      
    tendencia = polyval(coef, x)

    residuo_reg_lin = ((serie_precos["Adj Close"] - tendencia) / tendencia) * 100
    residuo_reg_lin = residuo_reg_lin.iloc[-1]
    
    # Beta
    beta = serie_precos["Adj Close"].pct_change().cov(benchmark["Adj Close"]) / benchmark["Adj Close"].var()
    
    # Retorno agrupado por mês
    serie_precos['ret'] = serie_precos["Adj Close"].pct_change()
    serie_precos['mes'] = serie_precos.index.month
    serie_precos['ano'] = serie_precos.index.year

    tabela = (
        serie_precos.groupby(['ano', 'mes'])[['ret']]
        .sum()
        .reset_index()
        .pivot(index='ano', columns='mes', values='ret')
    ) * 100
    
    data[codigo] = {
        "Ativo Total": ativo_total,
        "Resultado Bruto": resultado_bruto,
        "Receita de Vendas": receita_de_vendas,
        "Drawdown Atual": drawdown_atual,
        "Resíduo Regressão Linear": residuo_reg_lin,
        "Beta": beta,
        f"Retorno Agrupado por Mês (Média Mês:{date.today().month})": tabela[date.today().month].mean(skipna=True),
        f"Retorno Agrupado por Mês (Média Mês:{(date.today() + relativedelta(months=1)).month})": tabela[(date.today() + relativedelta(months=1)).month].mean(skipna=True),
        }
    
    

In [8]:
df_final = (
    df_b3_segmentos_setoriais.set_index("Código")
    .join(DataFrame.from_dict(data, orient="index"))
    .reset_index()
)

benchmark_row = DataFrame([{
    "Código": "^BVSP",
    "Ativo Total": nan,
    "Resultado Bruto": nan,
    "Receita de Vendas": nan,
    "Drawdown Atual": data["^BVSP"]["Drawdown Atual"],
    "Resíduo Regressão Linear": data["^BVSP"]["Resíduo Regressão Linear"],
    "Beta": data["^BVSP"]["Beta"],
    f"Retorno Agrupado por Mês (Média Mês:{date.today().month})": data["^BVSP"][f"Retorno Agrupado por Mês (Média Mês:{date.today().month})"],
    f"Retorno Agrupado por Mês (Média Mês:{(date.today() + relativedelta(months=1)).month})": data["^BVSP"][f"Retorno Agrupado por Mês (Média Mês:{(date.today() + relativedelta(months=1)).month})"]
}])

df_final = concat([df_final, benchmark_row], ignore_index=True)

set_option("display.max_rows", None)
df_final =  df_final.sort_values(by="Drawdown Atual", ascending=False)
df_final

,Código,Indice,Data Referencia,Ação,Tipo,Qtde. Teórica,Part. (%),Ativo Total,Resultado Bruto,Receita de Vendas,Drawdown Atual,Resíduo Regressão Linear,Beta,Retorno Agrupado por Mês (Média Mês:7),Retorno Agrupado por Mês (Média Mês:8)
48,PSSA3,IBEP,2026-07-13,PORTO SEGURO,ON NM,1.788254e+08,0.459,0.056534,NaN,NaN,0.000000,40.029621,0.540173,3.570144,5.421265
62,UGPA3,IBEP,2026-07-13,ULTRAPAR,ON NM,1.067870e+09,1.532,0.045903,NaN,NaN,0.000000,53.355377,1.194837,0.838524,0.262490
26,ENEV3,IBEP,2026-07-13,ENEVA,ON NM,1.913352e+09,2.462,0.041386,-0.340257,-0.268461,-0.144983,44.352006,0.751190,5.329783,4.708633
66,VBBR3,IBEP,2026-07-13,VIBRA,ON NM,1.192005e+09,1.837,0.043912,0.248137,-0.049013,-1.952617,54.981899,1.170716,4.725219,5.750551
31,GOAU4,IBEP,2026-07-13,GERDAU MET,PN N1,8.308378e+08,0.397,-0.011312,0.288102,-0.209715,-3.405863,14.708858,1.190707,8.970152,3.311804
36,ITSA4,IBEP,2026-07-13,ITAUSA,PN N1,5.970738e+09,3.952,0.028099,NaN,NaN,-4.342639,46.569249,0.969158,3.729873,2.173560
35,ISAE4,IBEP,2026-07-13,ISA ENERGIA,PN N1,4.155688e+08,0.582,0.013029,-0.013623,-0.173816,-5.139295,17.075947,0.487459,1.578116,0.623695
59,TAEE11,IBEP,2026-07-13,TAESA,UNT N2,2.185682e+08,0.426,0.035111,0.030653,0.020434,-5.657611,10.044863,0.449992,4.010960,1.427711
30,GGBR4,IBEP,2026-07-13,GERDAU,PN N1,1.258208e+09,1.352,-0.011312,0.288102,-0.209715,-6.653142,7.615753,1.116243,8.082939,1.395284
18,CPLE3,IBEP,2026-07-13,COPEL,ON NM,2.969407e+09,2.154,0.066665,-0.061209,-0.055854,-6.865520,41.238568,0.837125,3.075152,2.515007


In [6]:
df_final[df_final["Part. (%)"] > 5]

,Código,Indice,Data Referencia,Ação,Tipo,Qtde. Teórica,Part. (%),Ativo Total,Resultado Bruto,Receita de Vendas,Drawdown Atual,Resíduo Regressão Linear,Beta,Retorno Agrupado por Mês (Média Mês:7),Retorno Agrupado por Mês (Média Mês:8)
37,ITUB4,IBEP,2026-07-13,ITAUUNIBANCO,PN N1,5.349695e+09,11.069,0.011946,0.422955,0.098618,-8.519106,43.879179,0.998299,2.945274,2.729987
55,SBSP3,IBEP,2026-07-13,SABESP,ON NM,3.506733e+09,5.095,0.108769,0.020655,-0.116366,-11.639399,51.459778,0.966768,4.830103,-0.269287
64,VALE3,IBEP,2026-07-13,VALE,ON NM,4.268647e+09,14.789,-0.031397,-0.242255,-0.180473,-17.660114,3.115167,0.919426,2.301236,-1.171640
4,AXIA3,IBEP,2026-07-13,AXIA ENERGIA,ON NM,2.156760e+09,5.443,0.006771,0.000000,0.000000,-19.538351,13.432179,1.303794,8.393941,8.475445
